# CS3807 – Deep Learning Laboratory
# Experiment 4: Comparative Study of Deep CNN Architectures Using Transfer Learning

This notebook follows the lab manual section by section. Run all cells top to bottom.

**Before running:** In Colab, go to `Runtime → Change runtime type → T4 GPU` (or any GPU) — CPU-only will be very slow.

All figures are saved at **600 DPI** into `/content/outputs/` as the notebook runs.


## 0. Setup

In [ ]:
import os
os.makedirs('/content/outputs', exist_ok=True)

import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", len(gpus) > 0, gpus)

SAVE_DIR = '/content/outputs'
DPI = 600
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
def savefig(fname):
    path = os.path.join(SAVE_DIR, fname)
    plt.savefig(path, dpi=DPI, bbox_inches='tight')
    print("Saved:", path)


## Task 1: Dataset Preparation
1. Load CIFAR-10 using TensorFlow/Keras.
2. Normalize pixel values to [0,1].
3. Display 10 sample images.
4. Print dimensions of train/test sets.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# 2. Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# 4. Print dimensions
print("Training images shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Testing images shape:", x_test.shape)
print("Testing labels shape:", y_test.shape)


In [ ]:
# 3. Display 10 sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(class_names[int(y_train[i])])
    ax.axis('off')
plt.suptitle("Figure 1: Sample CIFAR-10 Images")
plt.tight_layout()
savefig('fig1_sample_cifar10_images.png')
plt.show()


**Inference (Figure 1 — Sample CIFAR-10 images):** *[Fill in after running — 2-3 lines on what the sample images show, e.g. resolution, diversity of classes, visual difficulty of the classification task.]*

## Task 2: Transfer Learning — Model Setup

We use **VGG16** as the primary pretrained model for the mandatory Task 2–5 pipeline
(per Additional Exercise 1). Real ImageNet weights are downloaded from Keras Applications —
nothing here is mocked.

Steps performed (as per manual):
1. Load pretrained ImageNet weights.
2. Remove the original classification layer (`include_top=False`).
3. Freeze the convolutional base.
4. Add a Global Average Pooling layer.
5. Add a Dense layer with ReLU activation.
6. Add the output layer with Softmax activation.

CIFAR-10 images are 32×32, which is too small for these architectures to extract useful
features, so a `Resizing` layer upsamples images to a larger size before they enter the
pretrained base.

In [ ]:
MODEL_CONFIGS = {
    'VGG16': dict(
        app=keras.applications.VGG16,
        preprocess=keras.applications.vgg16.preprocess_input,
        img_size=96,
    ),
    'ResNet50': dict(
        app=keras.applications.ResNet50,
        preprocess=keras.applications.resnet50.preprocess_input,
        img_size=96,
    ),
    'InceptionV3': dict(  # used later as a real stand-in for GoogleNet (Inception architecture)
        app=keras.applications.InceptionV3,
        preprocess=keras.applications.inception_v3.preprocess_input,
        img_size=96,
    ),
}

def build_transfer_model(name, dense_units=256, num_classes=10, trainable_base=False):
    cfg = MODEL_CONFIGS[name]
    base = cfg['app'](weights='imagenet', include_top=False,
                       input_shape=(cfg['img_size'], cfg['img_size'], 3))
    base.trainable = trainable_base

    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Resizing(cfg['img_size'], cfg['img_size'])(inputs)
    x = layers.Lambda(cfg['preprocess'])(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name=f'{name}_transfer')
    return model, base

def unfreeze_last_n_layers(base_model, n=20):
    """Task 4: unfreeze the last conv block (approximated as the last n layers)."""
    base_model.trainable = True
    for layer in base_model.layers[:-n]:
        layer.trainable = False
    return base_model

vgg_model, vgg_base = build_transfer_model('VGG16', dense_units=256)
vgg_model.summary()


## Task 3: Model Training (VGG16)

Optimizer: Adam, Learning Rate: 0.001, Batch Size: 32, Epochs: 12 (within the 10–20 range), Loss: Categorical Cross-Entropy.

In [ ]:
vgg_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_INITIAL = 12
start = time.time()
history_vgg = vgg_model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    batch_size=32,
    epochs=EPOCHS_INITIAL,
)
vgg_train_time_initial = time.time() - start
print(f"Initial training time: {vgg_train_time_initial:.1f}s")


In [ ]:
def plot_history(history, prefix, title_suffix=""):
    # Training & Validation Accuracy
    plt.figure(figsize=(7, 5))
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy')
    plt.title(f'Training & Validation Accuracy {title_suffix}')
    plt.legend(); plt.tight_layout()
    savefig(f'{prefix}_accuracy.png')
    plt.show()

    # Training & Validation Loss
    plt.figure(figsize=(7, 5))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(f'Training & Validation Loss {title_suffix}')
    plt.legend(); plt.tight_layout()
    savefig(f'{prefix}_loss.png')
    plt.show()

plot_history(history_vgg, 'fig2_vgg16_initial', '(VGG16, frozen base)')


**Inference (Figure 2 — VGG16 training/validation accuracy & loss, frozen base):** *[Fill in after running.]*

## Task 4: Fine-Tuning (VGG16)
1. Unfreeze the last convolution block.
2. Train for another 5–10 epochs.
3. Compare accuracy before vs after fine-tuning.

In [ ]:
acc_before_finetune = history_vgg.history['val_accuracy'][-1]

unfreeze_last_n_layers(vgg_base, n=4)  # VGG16 block5 = last 4 conv/pool layers

vgg_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_FINETUNE = 8
start = time.time()
history_vgg_ft = vgg_model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    batch_size=32,
    epochs=EPOCHS_FINETUNE,
)
vgg_train_time_finetune = time.time() - start
vgg_train_time_total = vgg_train_time_initial + vgg_train_time_finetune

acc_after_finetune = history_vgg_ft.history['val_accuracy'][-1]
print(f"Validation accuracy BEFORE fine-tuning: {acc_before_finetune:.4f}")
print(f"Validation accuracy AFTER fine-tuning:  {acc_after_finetune:.4f}")


In [ ]:
plot_history(history_vgg_ft, 'fig3_vgg16_finetune', '(VGG16, fine-tuned)')

plt.figure(figsize=(5, 5))
plt.bar(['Before Fine-tuning', 'After Fine-tuning'],
        [acc_before_finetune, acc_after_finetune])
plt.ylabel('Validation Accuracy')
plt.title('Figure 4: Accuracy Before vs After Fine-Tuning (VGG16)')
plt.tight_layout()
savefig('fig4_vgg16_before_after_finetune.png')
plt.show()


**Inference (Figures 3 & 4 — fine-tuning results):** *[Fill in after running.]*

## Task 5: Model Evaluation (VGG16)
Accuracy, Precision, Recall, F1-score, Confusion Matrix, Classification Report.

In [ ]:
y_pred_probs = vgg_model.predict(x_test, batch_size=64)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro')
rec = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Figure 5: Confusion Matrix (VGG16)')
plt.tight_layout()
savefig('fig5_vgg16_confusion_matrix.png')
plt.show()


**Inference (Figure 5 — Confusion Matrix):** *[Fill in after running.]*

In [ ]:
# Misclassified images (optional plot)
mis_idx = np.where(y_pred != y_true)[0]
sample_mis = np.random.choice(mis_idx, size=min(10, len(mis_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for ax, idx in zip(axes.flat, sample_mis):
    ax.imshow(x_test[idx])
    ax.set_title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=9)
    ax.axis('off')
plt.suptitle('Figure 6: Misclassified Images (VGG16)')
plt.tight_layout()
savefig('fig6_vgg16_misclassified.png')
plt.show()


**Inference (Figure 6 — Misclassified images):** *[Fill in after running.]*

## Section 16: Hyperparameter Study

The manual lists 6 hyperparameters (learning rate, batch size, epochs, optimizer,
dense units, frozen layers) with multiple values each. A full factorial grid would be
90+ separate training runs, which isn't practical on Colab. Instead this section runs a
**one-factor-at-a-time** study: start from a baseline config, vary one hyperparameter at
a time, and record validation accuracy. To keep runtime reasonable, each run uses a
5,000-image training subset and 3 epochs — this is a controlled comparison of settings,
not a final model (the final model is the one trained above in Tasks 3–4).

In [ ]:
x_sub = x_train[:5000]
y_sub = y_train_cat[:5000]

BASE_CONFIG = dict(lr=0.001, batch_size=32, optimizer='Adam', dense_units=256, frozen='All')

def run_config(lr, batch_size, optimizer_name, dense_units, frozen, epochs=3):
    model, base = build_transfer_model('VGG16', dense_units=dense_units)
    if frozen == 'Partial':
        unfreeze_last_n_layers(base, n=4)

    opt = keras.optimizers.Adam(learning_rate=lr) if optimizer_name == 'Adam' \
        else keras.optimizers.SGD(learning_rate=lr)

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    h = model.fit(x_sub, y_sub, validation_data=(x_test[:2000], y_test_cat[:2000]),
                  batch_size=batch_size, epochs=epochs, verbose=0)
    elapsed = time.time() - start
    val_acc = h.history['val_accuracy'][-1]
    return val_acc, elapsed

results = []

# Vary Learning Rate
for lr in [0.001, 0.0001]:
    va, t = run_config(lr, BASE_CONFIG['batch_size'], BASE_CONFIG['optimizer'],
                        BASE_CONFIG['dense_units'], BASE_CONFIG['frozen'])
    results.append(dict(factor='Learning Rate', value=lr, val_accuracy=va, time_s=t))

# Vary Batch Size
for bs in [16, 32, 64]:
    va, t = run_config(BASE_CONFIG['lr'], bs, BASE_CONFIG['optimizer'],
                        BASE_CONFIG['dense_units'], BASE_CONFIG['frozen'])
    results.append(dict(factor='Batch Size', value=bs, val_accuracy=va, time_s=t))

# Vary Optimizer
for opt_name in ['Adam', 'SGD']:
    va, t = run_config(BASE_CONFIG['lr'], BASE_CONFIG['batch_size'], opt_name,
                        BASE_CONFIG['dense_units'], BASE_CONFIG['frozen'])
    results.append(dict(factor='Optimizer', value=opt_name, val_accuracy=va, time_s=t))

# Vary Dense Units
for du in [128, 256]:
    va, t = run_config(BASE_CONFIG['lr'], BASE_CONFIG['batch_size'], BASE_CONFIG['optimizer'],
                        du, BASE_CONFIG['frozen'])
    results.append(dict(factor='Dense Units', value=du, val_accuracy=va, time_s=t))

# Vary Frozen Layers
for fr in ['All', 'Partial']:
    va, t = run_config(BASE_CONFIG['lr'], BASE_CONFIG['batch_size'], BASE_CONFIG['optimizer'],
                        BASE_CONFIG['dense_units'], fr)
    results.append(dict(factor='Frozen Layers', value=fr, val_accuracy=va, time_s=t))

import pandas as pd
hp_df = pd.DataFrame(results)
hp_df.to_csv(os.path.join(SAVE_DIR, 'hyperparameter_study.csv'), index=False)
hp_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('tab10', hp_df['factor'].nunique())
factor_colors = dict(zip(hp_df['factor'].unique(), colors))
bar_colors = [factor_colors[f] for f in hp_df['factor']]
labels = [f"{r.factor}\n{r.value}" for r in hp_df.itertuples()]
ax.bar(labels, hp_df['val_accuracy'], color=bar_colors)
ax.set_ylabel('Validation Accuracy')
ax.set_title('Figure 7: Hyperparameter Study — Validation Accuracy by Setting')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
savefig('fig7_hyperparameter_study.png')
plt.show()


**Inference (Figure 7 — Hyperparameter study):** *[Fill in after running.]*

## Additional Exercise 2: Repeat the Experiment Using ResNet50

Same pipeline as VGG16 (Tasks 2–5), using real pretrained ResNet50 ImageNet weights.

In [ ]:
resnet_model, resnet_base = build_transfer_model('ResNet50', dense_units=256)
resnet_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                      loss='categorical_crossentropy', metrics=['accuracy'])

start = time.time()
history_resnet = resnet_model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    batch_size=32, epochs=10
)
resnet_train_time = time.time() - start

y_pred_resnet = np.argmax(resnet_model.predict(x_test, batch_size=64), axis=1)
resnet_acc = accuracy_score(y_true, y_pred_resnet)
print(f"ResNet50 test accuracy: {resnet_acc:.4f}")
print(f"ResNet50 training time: {resnet_train_time:.1f}s")


In [ ]:
plot_history(history_resnet, 'fig8_resnet50', '(ResNet50)')

plt.figure(figsize=(5, 5))
plt.bar(['VGG16', 'ResNet50'], [acc_after_finetune, resnet_acc])
plt.ylabel('Test Accuracy')
plt.title('Figure 9: VGG16 vs ResNet50 (Additional Exercise 2)')
plt.tight_layout()
savefig('fig9_vgg16_vs_resnet50.png')
plt.show()


**Inference (Figures 8 & 9 — ResNet50 vs VGG16):** *[Fill in after running.]*

## Additional Exercise 6 / Section 18.2: LeNet-5, AlexNet, GoogleNet from Scratch/Pretrained

To genuinely compare all 5 architectures (per the Objectives and Additional Exercise 6:
*"Compare the performance of LeNet, AlexNet and ResNet"*), we build and train real versions
of LeNet-5 and an adapted AlexNet from scratch on CIFAR-10, and use real pretrained
**InceptionV3** weights as the GoogleNet (Inception) representative. VGG16 and ResNet50
numbers come from the runs above.

**Note on AlexNet:** the original AlexNet was designed for 227×227×3 ImageNet images. Training
it at that resolution on CIFAR-10 (32×32 native) would need heavy upsampling and a lot more
compute/time. We use a commonly-adapted, correctly-proportioned AlexNet-style architecture
(same layer philosophy: large early filters, ReLU, dropout, deep FC layers) scaled to CIFAR-10's
32×32 input, so the comparison stays runnable on Colab.

In [ ]:
def build_lenet5(num_classes=10):
    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(6, (5, 5), activation='tanh', padding='same')(inputs)
    x = layers.AveragePooling2D()(x)
    x = layers.Conv2D(16, (5, 5), activation='tanh')(x)
    x = layers.AveragePooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(120, activation='tanh')(x)
    x = layers.Dense(84, activation='tanh')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='LeNet5')

def build_alexnet_adapted(num_classes=10):
    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(96, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='AlexNet_adapted')

lenet = build_lenet5()
alexnet = build_alexnet_adapted()
lenet.summary()
alexnet.summary()


In [ ]:
def train_and_time(model, epochs, lr=0.001):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    h = model.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat),
                  batch_size=32, epochs=epochs)
    elapsed = time.time() - start
    y_pred_ = np.argmax(model.predict(x_test, batch_size=64), axis=1)
    acc_ = accuracy_score(y_true, y_pred_)
    return h, elapsed, acc_

history_lenet, lenet_time, lenet_acc = train_and_time(lenet, epochs=15)
history_alexnet, alexnet_time, alexnet_acc = train_and_time(alexnet, epochs=15)

print(f"LeNet-5  -> acc: {lenet_acc:.4f}, time: {lenet_time:.1f}s")
print(f"AlexNet  -> acc: {alexnet_acc:.4f}, time: {alexnet_time:.1f}s")


In [ ]:
plot_history(history_lenet, 'fig10_lenet5', '(LeNet-5)')
plot_history(history_alexnet, 'fig11_alexnet', '(AlexNet, adapted)')


**Inference (Figures 10 & 11 — LeNet-5 and AlexNet training curves):** *[Fill in after running.]*

In [ ]:
# GoogleNet stand-in: real pretrained InceptionV3
googlenet_model, googlenet_base = build_transfer_model('InceptionV3', dense_units=256)
googlenet_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                         loss='categorical_crossentropy', metrics=['accuracy'])

start = time.time()
history_googlenet = googlenet_model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    batch_size=32, epochs=8
)
googlenet_time = time.time() - start

y_pred_gn = np.argmax(googlenet_model.predict(x_test, batch_size=64), axis=1)
googlenet_acc = accuracy_score(y_true, y_pred_gn)
print(f"GoogleNet (InceptionV3) -> acc: {googlenet_acc:.4f}, time: {googlenet_time:.1f}s")


In [ ]:
plot_history(history_googlenet, 'fig12_googlenet', '(GoogleNet / InceptionV3)')


**Inference (Figure 12 — GoogleNet/InceptionV3 training curves):** *[Fill in after running.]*

## Section 18.2: Comparison of CNN Architectures (filled with actual results)

In [ ]:
comparison_df = pd.DataFrame([
    dict(Model='LeNet-5', Parameters=lenet.count_params(),
         Accuracy_pct=round(lenet_acc*100, 2), Training_Time_s=round(lenet_time, 1)),
    dict(Model='AlexNet (adapted)', Parameters=alexnet.count_params(),
         Accuracy_pct=round(alexnet_acc*100, 2), Training_Time_s=round(alexnet_time, 1)),
    dict(Model='VGG16', Parameters=vgg_model.count_params(),
         Accuracy_pct=round(acc_after_finetune*100, 2),
         Training_Time_s=round(vgg_train_time_total, 1)),
    dict(Model='GoogleNet (InceptionV3)', Parameters=googlenet_model.count_params(),
         Accuracy_pct=round(googlenet_acc*100, 2), Training_Time_s=round(googlenet_time, 1)),
    dict(Model='ResNet50', Parameters=resnet_model.count_params(),
         Accuracy_pct=round(resnet_acc*100, 2), Training_Time_s=round(resnet_train_time, 1)),
])
comparison_df.to_csv(os.path.join(SAVE_DIR, 'architecture_comparison.csv'), index=False)
comparison_df


In [ ]:
plt.figure(figsize=(9, 6))
plt.bar(comparison_df['Model'], comparison_df['Accuracy_pct'])
plt.ylabel('Test Accuracy (%)')
plt.title('Figure 13: Accuracy Comparison — LeNet-5, AlexNet, VGG16, GoogleNet, ResNet50')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
savefig('fig13_architecture_accuracy_comparison.png')
plt.show()


**Inference (Figure 13 — Architecture comparison):** *[Fill in after running.]*

## Section 19: Discussion Questions

**1. Why is AlexNet considered a breakthrough in deep learning?**
It was the first deep CNN to win ImageNet by a large margin (2012), showing that deep networks trained on GPUs with ReLU activations and Dropout could dramatically outperform traditional hand-engineered computer-vision pipelines, which shifted the whole field toward deep learning.

**2. Why does VGG16 use only 3×3 convolution filters?**
Stacking multiple 3×3 filters gives the same effective receptive field as one larger filter (e.g., two 3×3 layers ≈ one 5×5) while using fewer parameters and adding more non-linear activations in between, which improves the network's representational power.

**3. Explain the advantages of the Inception module.**
It applies multiple filter sizes (1×1, 3×3, 5×5) and pooling in parallel and concatenates the results, capturing features at multiple scales in one layer. 1×1 convolutions before the larger filters reduce dimensionality first, cutting parameters and computation while keeping accuracy high.

**4. What is the purpose of residual learning?**
Residual (skip) connections let the network learn a residual mapping F(x) = H(x) − x instead of the full mapping H(x) directly. This gives gradients a direct path back through the network, which prevents them from vanishing in very deep architectures and makes very deep networks trainable.

**5. Differentiate LeNet and ResNet.**
LeNet-5 (1998) is a shallow, ~60K-parameter network with only 2 conv layers, designed for small grayscale digit images. ResNet (2015) is dramatically deeper (50+ layers), uses residual/skip connections to avoid vanishing gradients, and is trained on large-scale color image datasets like ImageNet with millions of parameters.

**6. What is Transfer Learning?**
Transfer learning reuses a model already trained on a large dataset (e.g., ImageNet) as a starting point for a new, often smaller, dataset — reusing the learned low- and mid-level feature extractors instead of learning them from scratch.

**7. Why is fine tuning required?**
Freezing the pretrained base only trains the new classifier head, so the features stay generic to the original dataset. Fine-tuning unfreezes some of the later convolutional layers so they can adapt their higher-level features specifically to the new dataset, usually improving accuracy further.

**8. Explain the difference between dilated convolution and transpose convolution.**
Dilated (atrous) convolution inserts gaps between kernel elements to enlarge the receptive field without adding parameters or reducing resolution — used to capture more context, e.g., in segmentation. Transpose convolution does learnable *upsampling*, increasing the spatial size of a feature map — used to reconstruct/generate images, e.g., in autoencoders and GANs.

**9. Why do pretrained models converge faster?**
Their weights already encode general, reusable visual features (edges, textures, shapes) learned from a huge dataset, so the model starts much closer to a good solution than random initialization would, needing far fewer updates to adapt to the new task.

**10. Compare the computational complexity of LeNet and ResNet.**
LeNet-5 is extremely lightweight (~60K parameters, 2 conv layers), so it trains and runs almost instantly even on a CPU. ResNet50 has ~25.6M parameters across 50 layers, requiring substantially more FLOPs, memory, and (ideally) a GPU — but its residual connections make that extra depth actually trainable, which a naive 50-layer plain network without skip connections would struggle to be.

## Section 18.1: Performance Metrics (Primary Model — VGG16, fine-tuned)

In [ ]:
metrics_summary = pd.DataFrame([{
    'Training Accuracy': round(history_vgg_ft.history['accuracy'][-1], 4),
    'Testing Accuracy': round(acc, 4),
    'Precision': round(prec, 4),
    'Recall': round(rec, 4),
    'F1-score': round(f1, 4),
    'Training Time (s)': round(vgg_train_time_total, 1),
    'Total Parameters': vgg_model.count_params(),
}])
metrics_summary


## Summary

All figures (Figures 1–13) and CSVs (`hyperparameter_study.csv`, `architecture_comparison.csv`)
have been saved at 600 DPI to `/content/outputs/`. Once you've run the whole notebook, download
that folder (or zip it) and send me the results/plots — I'll help you write the inferences and
put together the lab report.

In [ ]:
# Optional: zip all outputs for easy download
import shutil
shutil.make_archive('/content/experiment4_outputs', 'zip', SAVE_DIR)
print("Zipped outputs to /content/experiment4_outputs.zip")
